# qlora-eval — base vs LoRA vs QLoRA on a strict-JSON triage task

**Goal:** measure whether parameter-efficient fine-tuning actually improves a small model on a well-defined task - and what it costs in VRAM and minutes.

**Protocol**
- Task: support-ticket triage -> strict JSON `{intent, priority, escalate}`.
- Model: `Qwen/Qwen2.5-0.5B-Instruct` (fits a free Colab T4).
- Data: synthetic, seeded, 400 train / 100 eval; the same eval set for every config.
- Configs: zero-shot base, LoRA (fp16), QLoRA (4-bit NF4) - same steps, seed, data.
- Metrics: JSON validity, exact match, field accuracy, eval wall-clock, peak VRAM.
- No cherry-picking: eval examples are generated before any training.


In [ ]:
%pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate" "bitsandbytes>=0.44"


In [ ]:
import gc, json, random, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
N_TRAIN, N_EVAL = 400, 100
MAX_STEPS, LR = 150, 2e-4
BATCH_SIZE, GRAD_ACCUM = 4, 2
MAX_LEN, EVAL_BATCH = 256, 8

random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
INTENTS = ["billing", "technical", "account", "shipping", "refund", "feature_request"]
TEMPLATES = {
    "billing": ["I was charged twice for my subscription.", "My invoice shows an amount I do not recognise.", "The payment failed but I still got billed.", "Please explain the extra charge on my card."],
    "technical": ["The app crashes every time I open the dashboard.", "Sync is stuck and nothing updates.", "I get an error page after logging in.", "The export button does nothing."],
    "account": ["I need to reset my password.", "Please change the email on my account.", "I cannot enable two-factor authentication.", "My profile settings will not save."],
    "shipping": ["My order has not arrived yet.", "The tracking page has not moved in five days.", "The package was delivered to the wrong address.", "Can I change the delivery address?"],
    "refund": ["I want a refund for last month.", "The refund I was promised never arrived.", "Please cancel and refund my annual plan.", "I returned the item and need my money back."],
    "feature_request": ["Please add dark mode to the mobile app.", "It would help to export reports as CSV.", "Can you support multiple languages?", "I would like an API webhook for new orders."],
}
CUES = {
    "high": ["This is urgent.", "Please handle this ASAP.", "This is critical for us.", "I need this fixed right now."],
    "medium": ["Please look into it soon.", "I need an answer today.", "Can you fix this this week?"],
    "low": ["No rush at all.", "Whenever you can.", "This can wait until next month."],
}
ESCALATE_INTENTS = {"technical", "refund"}

def make_example(rng):
    intent = rng.choice(INTENTS)
    priority = rng.choices(["high", "medium", "low"], weights=[3, 4, 3])[0]
    text = rng.choice(TEMPLATES[intent]) + " " + rng.choice(CUES[priority])
    target = {"intent": intent, "priority": priority, "escalate": priority == "high" and intent in ESCALATE_INTENTS}
    return {"text": text, "target": target}

train = [make_example(random.Random(SEED + i)) for i in range(N_TRAIN)]
evalset = [make_example(random.Random(SEED + 10_000 + i)) for i in range(N_EVAL)]
print(json.dumps(train[0], indent=2))
print(json.dumps(evalset[0], indent=2))


In [ ]:
SYSTEM = ("You are a support-ticket triage assistant. Return ONLY a JSON object with keys "
          "\"intent\" (billing|technical|account|shipping|refund|feature_request), "
          "\"priority\" (high|medium|low), \"escalate\" (boolean). No prose.")

def render_prompt(tokenizer, text):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def target_text(example):
    return json.dumps(example["target"], ensure_ascii=False)

JSON_RE = re.compile(r"\{.*?\}", re.S)

def parse_prediction(text):
    match = JSON_RE.search(text)
    if match is None:
        return None
    try:
        obj = json.loads(match.group(0))
    except Exception:
        return None
    return obj if isinstance(obj, dict) else None

def score(examples, raw_predictions):
    valid = exact = 0
    field_ok = {"intent": 0, "priority": 0, "escalate": 0}
    for example, raw in zip(examples, raw_predictions):
        pred = parse_prediction(raw)
        if pred is None:
            continue
        valid += 1
        for key in field_ok:
            if pred.get(key) == example["target"][key]:
                field_ok[key] += 1
        if all(pred.get(k) == example["target"][k] for k in example["target"]):
            exact += 1
    n = len(examples)
    metrics = {
        "n": n,
        "json_valid": valid / n,
        "exact_match": exact / n,
        "field_accuracy": sum(field_ok.values()) / (3 * n),
    }
    metrics.update({f"acc_{key}": value / n for key, value in field_ok.items()})
    return metrics


In [ ]:
def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def load_base(quantized=False):
    kwargs = {"device_map": "auto", "torch_dtype": torch.float16}
    if quantized:
        kwargs.pop("torch_dtype")
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)

@torch.no_grad()
def generate_predictions(model, tokenizer, examples, batch_size=EVAL_BATCH):
    model.eval()
    predictions = []
    for start in range(0, len(examples), batch_size):
        batch = examples[start:start + batch_size]
        prompts = [render_prompt(tokenizer, example["text"]) for example in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        generated = out[:, enc.input_ids.shape[1]:]
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return predictions

def evaluate(model, tokenizer, examples, label):
    started = time.time()
    predictions = generate_predictions(model, tokenizer, examples)
    elapsed = time.time() - started
    metrics = score(examples, predictions)
    metrics["eval_seconds"] = round(elapsed, 1)
    metrics["config"] = label
    print(label, json.dumps(metrics, indent=2))
    return metrics


In [ ]:
tokenizer = load_tokenizer()
base_model = load_base()
baseline_metrics = evaluate(base_model, tokenizer, evalset, "base")
del base_model
gc.collect(); torch.cuda.empty_cache()
print("peak vram after base eval (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))


In [ ]:
def encode(example, tokenizer):
    prompt = render_prompt(tokenizer, example["text"])
    answer = target_text(example)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]
    return {"input_ids": input_ids, "labels": labels}

def make_collator(tokenizer):
    def collate(batch):
        longest = max(len(item["input_ids"]) for item in batch)
        input_ids, labels, attention = [], [], []
        for item in batch:
            pad = longest - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad)
            labels.append(item["labels"] + [-100] * pad)
            attention.append([1] * len(item["input_ids"]) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attention),
        }
    return collate

def train_adapter(quantized=False, steps=MAX_STEPS):
    gc.collect(); torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    tokenizer = load_tokenizer()
    model = load_base(quantized=quantized)
    if quantized:
        model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)
    model.train()
    encoded = [encode(example, tokenizer) for example in train]
    collate = make_collator(tokenizer)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min(1.0, (s + 1) / 10))
    rng = random.Random(SEED)
    started = time.time()
    step = 0
    while step < steps:
        batch = collate([encoded[i] for i in rng.sample(range(len(encoded)), BATCH_SIZE)])
        batch = {key: value.to(model.device) for key, value in batch.items()}
        loss = model(**batch).loss / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        step += 1
        if step % 50 == 0:
            print("step", step, "loss", round(float(loss) * GRAD_ACCUM, 4))
    train_seconds = time.time() - started
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    label = "qlora" if quantized else "lora"
    metrics = evaluate(model, tokenizer, evalset, label)
    metrics.update({"train_seconds": round(train_seconds, 1), "peak_vram_gb": round(peak, 2), "steps": steps})
    return metrics


In [ ]:
lora_metrics = train_adapter(quantized=False)
gc.collect(); torch.cuda.empty_cache()


In [ ]:
qlora_metrics = train_adapter(quantized=True)
gc.collect(); torch.cuda.empty_cache()


In [ ]:
import pandas as pd

runs = [baseline_metrics, lora_metrics, qlora_metrics]
columns = ["config", "json_valid", "exact_match", "field_accuracy", "acc_intent", "acc_priority",
           "acc_escalate", "eval_seconds", "train_seconds", "peak_vram_gb", "steps"]
table = pd.DataFrame([{column: run.get(column) for column in columns} for run in runs])
display(table)

results = {
    "protocol": {"model": MODEL_NAME, "n_train": N_TRAIN, "n_eval": N_EVAL,
                 "max_steps": MAX_STEPS, "lr": LR, "batch_size": BATCH_SIZE,
                 "grad_accum": GRAD_ACCUM, "seed": SEED},
    "runs": runs,
}
with open("qlora_eval_results.json", "w") as handle:
    json.dump(results, handle, indent=2)
print(table.to_markdown(index=False))
print()
print("Saved qlora_eval_results.json - download it and commit it to the repo.")


## How to read the results

- **json_valid** - does the model produce parseable JSON at all?
- **exact_match / field_accuracy** - task quality; compare each adapter against base.
- **eval_seconds** - inference wall-clock on the same 100 examples.
- **train_seconds / peak_vram_gb** - the cost side of the trade-off.

Write-up question: **does QLoRA match LoRA quality at lower VRAM, and how much time does that cost?**
The honest answer belongs in the README (including cases where nothing improved).
